<a href="https://colab.research.google.com/github/varshil009/ai-engineering/blob/main/DecoderConstraints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup: Install Libraries and Download Model

First, we need to install the necessary libraries. For `llama-cpp-python` to utilize the T4 GPU, we'll install it with CUDA support. Then we'll download a quantized GGUF model from Hugging Face.

In [1]:
# Install other required libraries
!pip install transformers pydantic

In [3]:
!pip uninstall -y llama-cpp-python

!pip install --upgrade --force-reinstall \
  llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

Found existing installation: llama_cpp_python 0.3.32
Uninstalling llama_cpp_python-0.3.32:
  Successfully uninstalled llama_cpp_python-0.3.32
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 543.7 kB/s eta 0:00:00
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached typing_extensions-4.16.0-py

### Download a Quantized Hugging Face Model

We'll use a `Mistral-7B-Instruct-v0.2` GGUF model, which is a good balance between capability and size for a T4 GPU when quantized. We'll download an `q4_K_M` quantized version.

In [21]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
import os
from huggingface_hub import hf_hub_download

# Define the model to download
model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_file = "mistral-7b-instruct-v0.2.Q4_K_M.gguf" # A good quantization for T4

# Define local path for the model
model_path = os.path.join(".", model_file)

# Download the model if it doesn't exist
if not os.path.exists(model_path):
    print(f"Downloading {model_file} from Hugging Face Hub...")
    hf_hub_download(
        repo_id=model_name,
        filename=model_file,
        local_dir=".",
        local_dir_use_symlinks=False,
    )
    print("Download complete.")
else:
    print(f"{model_file} already exists locally.")


mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

Download complete.


## 2. Setup `llama.cpp` for Inference

Now we'll load the downloaded GGUF model using `Llama.cpp` and prepare it for inference. We'll enable `n_gpu_layers` to offload layers to the GPU, which is crucial for performance on a T4.

In [6]:
from llama_cpp import Llama

# Initialize the Llama model
# n_gpu_layers=-1 attempts to offload all layers to the GPU.
# For T4, a 7B Q4_K_M model should fit well.
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1, # Offload all layers to GPU (adjust if OOM errors occur)
    n_ctx=4096,      # Context window size
    verbose=False    # Suppress llama.cpp verbose output
)

print("Llama model loaded successfully with GPU acceleration.")

Llama model loaded successfully with GPU acceleration.


## 3. Define Pydantic Schema for Validation

We'll define a simple Pydantic model to represent the desired JSON output structure. This will be used to validate the LLM's output.

In [25]:
from pydantic import BaseModel, Field
from typing import List, Optional

class UserProfile(BaseModel):
    name: str = Field(description="The user's full name")
    age: int = Field(description="The user's age")
    email: str = Field(description="The user's email address")
    interests: List[str] = Field(description="A list of the user's hobbies and interests")
    is_premium: bool = Field(description="Whether the user has a premium account")

class UserResponse(BaseModel):
    user: UserProfile = Field(description="Details about the user")
    message: str = Field(description="A concluding message about the user")

print("Pydantic schema defined: UserResponse")

Pydantic schema defined: UserResponse


## 4. Generate JSON with Grammar Constraint and Validate

Now, we'll use `llama-cpp-python`'s built-in JSON grammar feature. This will constrain the LLM's output to strictly adhere to the JSON schema derived from our Pydantic model. After generation, we'll validate the output using the Pydantic model.

In [10]:
import json
from llama_cpp.llama_grammar import LlamaGrammar

# Convert Pydantic schema to GBNF grammar for llama.cpp
json_grammar = LlamaGrammar.from_json_schema(UserResponse.schema_json())

# Prompt for JSON generation
prompt_valid_json = """### Instruction:
Generate a JSON object describing a user based on the following information: Name: Alice Wonderland, Age: 30, Email: alice@example.com, Interests: reading, hiking, coding. Alice has a premium account. Include a short concluding message.

### Response:
"""

print("Generating valid JSON output...")

# Generate text with JSON grammar constraint
response_valid = llm.create_completion(
    prompt=prompt_valid_json,
    max_tokens=500,
    temperature=0.7,
    grammar=json_grammar # Apply the JSON grammar constraint
)

# Extract the generated JSON string
generated_json_str = response_valid["choices"][0]["text"].strip()

print("\n--- Generated JSON Output (constrained) ---")
print(generated_json_str)

# Validate with Pydantic
try:
    validated_data = UserResponse.parse_raw(generated_json_str)
    print("\n--- Pydantic Validation Successful! ---")
    print(validated_data.json(indent=2))
except Exception as e:
    print(f"\n!!! Pydantic Validation Failed: {e} !!!")


/tmp/ipykernel_17560/753302413.py:5: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  json_grammar = LlamaGrammar.from_json_schema(UserResponse.schema_json())


Generating valid JSON output...

--- Generated JSON Output (constrained) ---
{ "user": { "name": "Alice Wonderland", "age": 30, "email": "alice@example.com", "interests": ["reading", "hiking", "coding"], "is_premium": true } ,"message": "This is a JSON representation of user Alice Wonderland, who is 30 years old, has the email address alice@example.com, enjoys reading, hiking, and coding, and holds a premium account." }

--- Pydantic Validation Successful! ---

!!! Pydantic Validation Failed: `dumps_kwargs` keyword arguments are no longer supported. !!!


/tmp/ipykernel_17560/753302413.py:32: PydanticDeprecatedSince20: The `parse_raw` method is deprecated; if your data is JSON use `model_validate_json`, otherwise load the data then use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  validated_data = UserResponse.parse_raw(generated_json_str)
/tmp/ipykernel_17560/753302413.py:34: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(validated_data.json(indent=2))


In [12]:
validated_data.json()

/tmp/ipykernel_17560/364547507.py:1: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  validated_data.json()


'{"user":{"name":"Alice Wonderland","age":30,"email":"alice@example.com","interests":["reading","hiking","coding"],"is_premium":true},"message":"This is a JSON representation of user Alice Wonderland, who is 30 years old, has the email address alice@example.com, enjoys reading, hiking, and coding, and holds a premium account."}'

## 5. Generate Faulty Grammar

Now, let's try to prompt the model to generate a faulty grammar (or at least something that doesn't conform to our schema) by explicitly asking it to generate invalid JSON or providing a conflicting instruction. For this, we'll *remove* the grammar constraint to allow the model to deviate.

In [9]:
# Prompt for faulty JSON generation (without grammar constraint)
prompt_faulty_json = """### Instruction:
Generate a JSON object about a user named Bob. Make sure to intentionally miss the closing brace '}' for the 'user' object and add an extra comma at the end of the interests list. Name: Bob, Age: 25, Email: bob@example.com, Interests: gaming, watching movies.

### Response:
"""

print("Generating faulty JSON output...")

# Generate text *without* JSON grammar constraint to allow errors
response_faulty = llm.create_completion(
    prompt=prompt_faulty_json,
    max_tokens=500,
    temperature=0.7,
    # grammar=json_grammar # IMPORTANT: Do NOT apply grammar for faulty generation
)

# Extract the generated string
generated_faulty_str = response_faulty["choices"][0]["text"].strip()

print("\n--- Generated Faulty Output ---")
print(generated_faulty_str)

# Attempt to validate with Pydantic
try:
    validated_data_faulty = UserResponse.parse_raw(generated_faulty_str)
    print("\n!!! Pydantic Validation Unexpectedly Successful (Model might have corrected itself or output was not faulty enough) !!!")
    print(validated_data_faulty.json(indent=2))
except Exception as e:
    print(f"\n--- Pydantic Validation Failed (as expected): {e} ---")

# Also try json.loads to see if it's even valid JSON
try:
    json.loads(generated_faulty_str)
    print("\n--- json.loads was successful (Output is valid JSON, but might not match schema) ---")
except json.JSONDecodeError as e:
    print(f"\n--- json.loads failed (Output is not even valid JSON): {e} ---")



Generating faulty JSON output...

--- Generated Faulty Output ---
{
  "Name": "Bob",
  "Age": 25,
  "Email": "bob@example.com",
  "Interests": [
    "gaming",
    "watching movies"
  ],
  // Missing closing brace
}

// With the extra comma added at the end:
{
  "Name": "Bob",
  "Age": 25,
  "Email": "bob@example.com",
  "Interests": [
    "gaming",
    "watching movies"
  ],
  // Missing closing brace
  // Extra comma
}

--- Pydantic Validation Failed (as expected): 1 validation error for UserResponse
__root__
  Expecting property name enclosed in double quotes: line 9 column 3 (char 122) [type=value_error.jsondecode, input_value='{\n  "Name": "Bob",\n  "...ce\n  // Extra comma\n}', input_type=str] ---

--- json.loads failed (Output is not even valid JSON): Expecting property name enclosed in double quotes: line 9 column 3 (char 122) ---


/tmp/ipykernel_17560/1552829188.py:26: PydanticDeprecatedSince20: The `parse_raw` method is deprecated; if your data is JSON use `model_validate_json`, otherwise load the data then use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  validated_data_faulty = UserResponse.parse_raw(generated_faulty_str)


## Re-Attempt using validation error

In [13]:
str({"hello" : 0})

"{'hello': 0}"

In [32]:
# Prompt for faulty JSON generation (without grammar constraint)
prompt_faulty_json = """### Instruction:
Generate a JSON object about a user named Bob. Name: Bob, Age: 25, Email: bob@example.com, Interests: gaming, watching movies.

### Response:
"""

print("Generating faulty JSON output...")

# Generate text *without* JSON grammar constraint to allow errors
response_faulty = llm.create_completion(
    prompt=prompt_faulty_json,
    max_tokens=500,
    temperature=0.7,
    # grammar=json_grammar # IMPORTANT: Do NOT apply grammar for faulty generation
)

# Extract the generated string
generated_faulty_str = response_faulty["choices"][0]["text"].strip()

print("\n--- Generated Faulty Output ---")
print(generated_faulty_str)

error = False
# Attempt to validate with Pydantic
try:
    validated_data_faulty = UserResponse.parse_raw(generated_faulty_str[7:-3])
    print("\n!!! Pydantic Validation Unexpectedly Successful (Model might have corrected itself or output was not faulty enough) !!!")
    print(validated_data_faulty.json(indent=2))
except Exception as e:
    error = {"input" : "for the 'user' object create a json response on given information. Name: Bob, Age: 25, Email: bob@example.com, Interests: gaming, watching movies.",
             "exception" : e,
             "grammar" : "strictly parsable json string with given format"}
    print(e)
    print("____________________________________________________________________")

if error:
    retry_prompt = llm.create_completion(
                    prompt=str(error)+'add is_premium attribute to user',
                    max_tokens=500,
                    temperature=0.7,
                    grammar=json_grammar
                )
    retry_answer = retry_prompt["choices"][0]["text"].strip()
# Also try json.loads to see if it's even valid JSON
try:
    json.loads(retry_answer)
    print("\n--- json.loads was successful, valid json file ---")
    if UserResponse.parse_raw(retry_answer):
        print("\n---PyDantic validation passed ---")
    else:
        print("\n---PyDantic validation failed ---")

except json.JSONDecodeError as e:
    print(f"\n--- json.loads failed (Output is not even valid JSON): {e} ---")


Generating faulty JSON output...

--- Generated Faulty Output ---
```json
{
  "name": "Bob",
  "age": 25,
  "email": "bob@example.com",
  "interests": ["gaming", "watching movies"]
}
```
2 validation errors for UserResponse
user
  Field required [type=missing, input_value={'name': 'Bob', 'age': 25...ng', 'watching movies']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
message
  Field required [type=missing, input_value={'name': 'Bob', 'age': 25...ng', 'watching movies']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
____________________________________________________________________

--- json.loads was successful, valid json file ---

---PyDantic validation passed ---


### Correcting Faulty Output Parsing to Reveal Pydantic Errors

Let's extract the actual JSON string from the markdown fences generated by the LLM in the previous cell to see the Pydantic validation errors more clearly.

In [31]:
import re

# Extract JSON from markdown fences
json_match = re.search(r'```json\s*([\s\S]*?)\s*```', generated_faulty_str)

cleaned_faulty_json_str = ''
if json_match:
    cleaned_faulty_json_str = json_match.group(1).strip()
    print("\n--- Extracted Clean JSON ---")
    print(cleaned_faulty_json_str)
else:
    # If no markdown, assume the entire string is the JSON (or try to)
    cleaned_faulty_json_str = generated_faulty_str.strip()
    print("\n--- No markdown fences found, using raw string ---")
    print(cleaned_faulty_json_str)

# Now, attempt to validate the cleaned JSON with Pydantic
try:
    validated_data_cleaned = UserResponse.model_validate_json(cleaned_faulty_json_str)
    print("\n!!! Pydantic Validation Unexpectedly Successful with cleaned data (Model might have corrected itself or output was not faulty enough) !!!")
    print(validated_data_cleaned.model_dump_json(indent=2))
except Exception as e:
    print(f"\n--- Pydantic Validation Failed (as expected, likely missing 'is_premium' or 'message'): {e} ---")

# Check if it's valid JSON using json.loads()
try:
    json.loads(cleaned_faulty_json_str)
    print("\n--- json.loads was successful (Cleaned output is valid JSON) ---")
except json.JSONDecodeError as e:
    print(f"\n--- json.loads failed (Cleaned output is NOT valid JSON): {e} ---")



--- Extracted Clean JSON ---
{
  "name": "Bob",
  "age": 25,
  "email": "bob@example.com",
  "interests": ["gaming", "watching movies"]
}

--- Pydantic Validation Failed (as expected, likely missing 'is_premium' or 'message'): 2 validation errors for UserResponse
user
  Field required [type=missing, input_value={'name': 'Bob', 'age': 25...ng', 'watching movies']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
message
  Field required [type=missing, input_value={'name': 'Bob', 'age': 25...ng', 'watching movies']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing ---

--- json.loads was successful (Cleaned output is valid JSON) ---


In [28]:
retry_answer

'{ "user" : { "name" : "Bob", "age" : 25, "email" : "bob@example.com", "interests" : ["gaming", "watching movies"], "is_premium" : true } , "message" : "Successfully created JSON response for UserResponse" }'

In [22]:
try :
    UserResponse.parse_raw('{ "user": { "name": "Bob", "age": 25, "interests": ["gaming", "watching movies"] } ,"message": "UserResponse: The following fields are required: interests." }')
except Exception as e:
    print(e)

1 validation error for UserResponse
user.email
  Field required [type=missing, input_value={'name': 'Bob', 'age': 25...ng', 'watching movies']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
